<!-- nav -->
[← 10 · Asking the store: mutations as queries](10_Asking_The_Store.ipynb) · [Home](../../README.md) · [12 · Keeping history: provenance, journals and snapshots →](12_Keeping_History.ipynb)

# 11 · A parts catalog: schemas that build on schemas

**The problem.** An electronics team keeps a catalog of parts. Every part has a name and a way it is mounted, through
holes or on the surface. A resistor is a part with a resistance; a chip is a part that mounts only on the surface. A
part's rating is a number or a short text, whichever its datasheet gives, and a bus is as wide as each design needs.
Copying a part's properties into a resistor's and a chip's schemas would let the copies drift apart; writing every
rating as text would lose its numbers; and one schema per bus width would multiply schemas without end.

This case study shows three ways a schema builds on others:

1. A **refinement** has the members of the schema it refines, and its own, and its objects are that schema's objects
   too ([CAP-167: schemas refined by other schemas](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-167.md)). It may redeclare a member, only narrower.
2. A **value of any schema** holds a value of one of the schemas it is restricted to, or of any schema at all
   ([CAP-157: values of any schema, restricted or not](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-157.md)).
3. **Parameters** stand for what each use of a schema binds
   ([CAP-171: parameters attached to every schema kind by extension](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-171.md)).

Last, the catalog goes to a store that has never heard of parameters, which reads what it can
([CAP-165: snapshots read by a store lacking their extensions](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-165.md)).

In [1]:
import { Builders, JSON as Json, Parametrics, Plain, Schemas, Validators } from "@mbse/schemas/Framework";

<a id="T11.1"></a>
## 11.1 · A refinement: a resistor is a part

The install defines a part, its mount, a union of two value objects (case study 9), and a catalog, a singleton that
lists parts. `cat.Resistor` says only what a resistor adds, a resistance, and `.refines([_.Part])`: the rest it has
from `cat.Part`, which it refines.

In [2]:
/** The catalog: parts, how each is mounted, a catalog listing them, and resistors, which are parts. */
function install(txn: any): any {
  const S = txn.Schemas; // the store's own schemas, by name
  return txn.uid("tutorial.catalog/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Through: S.OfObject, Surface: S.OfObject, Mount: S.OfUnion, Part: S.OfObject,
      Lists: S.OfRelation, Catalog: S.OfObject, Resistor: S.OfObject })
    .spec([
      (_: any) => _.Through.name("cat.Through").properties({ pins: S.OfInt }),
      (_: any) => _.Surface.name("cat.Surface").properties({ pitch: S.OfFloat }), // mm between pins
      (_: any) => _.Mount.name("cat.Mount").branches([
        (b: any) => b.name("through").schema([_.Through]),
        (b: any) => b.name("surface").schema([_.Surface]),
      ]),
      (_: any) => _.Part.name("cat.Part").ref(true).properties({ name: S.OfStr, mount: _.Mount }).adjacencies({
        catalog: (a: any) => a.relation([_.Lists]).me("part") }),
      (_: any) => _.Lists.name("cat.Lists").links([(t: any) => t.name("catalog"), (t: any) => t.name("part")]).uniques([]),
      (_: any) => _.Catalog.name("cat.Catalog").ref(true).singleton("cat.TheCatalog").adjacencies({
        parts: (a: any) => a.relation([_.Lists]).me("catalog") }),
      (_: any) => _.Resistor.name("cat.Resistor").refines([_.Part]).properties({ ohms: S.OfInt }),
    ])]))]);
}

const store: any = new Builders.Store();
store.transaction(install);
const [Part, Resistor] = [store.registered("cat.Part"), store.registered("cat.Resistor")];
console.log([...Resistor.properties.keys()], [...Resistor.adjacencies.keys()], Resistor.ref, Resistor.validate());
console.log(Schemas.is_a(Resistor, Part), Schemas.is_a(Part, Resistor));

[ "name", "mount", "ohms" ] [ "catalog" ] true []


true false


`cat.Resistor` has a part's name, mount and catalog, and its own `ohms`; it is a reference object schema, as what it
refines is. `Schemas.is_a` says a resistor is a part, and not the other way round. A member added to `cat.Part` later,
by another install, a resistor has too.

A resistor is built as any object is, and is a part wherever a part is wanted: it fills the catalog's `part` link,
and the store's extent of `cat.Part` holds it. Validate it by its own schema:

In [3]:
const catalog = store.singleton("cat.TheCatalog");
const r10k = store.transaction(() => {
  const r10k = store.cat.Resistor((r: any) => r.name("10k").ohms(10000n).mount((m: any) => m.through((t: any) => t.pins(2n))));
  store.cat.Catalog(catalog, (c: any) => c.parts([r10k]));
  return r10k;
});
console.log([...store.extent("cat.Part")].map((p: any) => `${p.name}`), r10k.mount.through.pins.get());
console.log(Validators.Validate(store)(Resistor, r10k));

[ "10k" ] 2n


[]


`mount((m: any) => m.through(...))` sets the union's `through` branch: a property whose value is a union has the union's
branches as members (case study 9 sets one through `as_union`, which this shortens).

<a id="T11.2"></a>
## 11.2 · Narrowing a member: a chip mounts on the surface

A chip is a part, but never mounted through holes. Its refinement redeclares `mount`, narrower: as `cat.Surface`, the
schema of the union's `surface` branch. Another install adds it, naming the schemas it builds on by their names
(`txn.cat.Part`), which the store holds already. A redeclared member must narrow the one it replaces: the same schema
or a refinement of it, one of a union's branches' schemas, a container whose index and element narrow, or one of the
schemas a value of any schema may hold. `validate()` reports one it can't confirm:

In [4]:
/** Chips: parts mounted on the surface only. */
function chips(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("tutorial.catalog.chips/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Chip: S.OfObject })
    .spec([(_: any) => _.Chip.name("cat.Chip").refines([txn.cat.Part]).properties({ mount: txn.cat.Surface })])]))]);
}

store.transaction(chips);
const Chip = store.registered("cat.Chip");
console.log([...Chip.properties.keys()], Chip.properties.get("mount").schema.name, Chip.validate());
store.transaction(() => {
  console.log(store.Schemas.OfObject((o: any) => o.refines([Part]).properties({ mount: store.Schemas.OfStr })).validate());
});

[ "name", "mount" ] cat.Surface []


[
  "property 'mount' cannot be confirmed to narrow the refined schema's"
]


A chip's `mount` is a surface mount itself, not a union holding one, so it is set as a value object is, and written
bare, where a part's is written under its branch's name. Each object is read and written by its own schema's members,
never guessed from the data. Here is what each one's document sets:

In [5]:
/** The properties an object's document sets, by name, as it writes them (case study 6). */
function written(schema: any, obj: any): Map<string, any> {
  const spec = (Plain.ToPlain(store)(schema, obj) as any).get("mutations")[0].get("upsert").get("exists")[0].get("spec")[0].get("0");
  return new Map(spec.filter((m: any) => m.has("as_property"))
    .map((m: any) => [m.get("as_property").get("name"), m.get("as_property").get("mutations")[0].get("set").get("value")]));
}

const mcu = store.transaction(() => {
  const mcu = store.cat.Chip((c: any) => c.name("MCU").mount((s: any) => s.pitch(0.5)));
  store.cat.Catalog(catalog, (c: any) => c.parts([mcu]));
  return mcu;
});
console.log([...store.extent("cat.Part")].map((p: any) => `${p.name}`), mcu.mount.pitch.get(), Validators.Validate(store)(Chip, mcu));
console.log(Json.dumps(written(Resistor, r10k)));
console.log(Json.dumps(written(Chip, mcu)));

[ "10k", "MCU" ] 0.5 []


{"name": "10k", "mount": {"through": {"pins": 2}}, "ohms": 10000}


{"name": "MCU", "mount": {"pitch": 0.5}}


<a id="T11.3"></a>
## 11.3 · Validating a refinement's object against what it refines

A resistor and a chip are parts, so each validates against `cat.Part` too, by its own schema
([CAP-167: schemas refined by other schemas](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-167.md)).
`Validators.Validate`, `Validators.Check` and `Reachable`, which also checks every object reachable from the one
given, each accept an object of the schema given or of a refinement of it. A chip is no resistor, so it is refused
against `cat.Resistor`:

In [6]:
console.log(Validators.Validate(store)(Part, r10k), Validators.Validate(store)(Part, mcu), Validators.Check(store)(Part, mcu).holds);
console.log(Validators.Validate(store).Reachable(Part, mcu)); // the chip, the catalog listing it, and the resistor
console.log(Validators.Validate(store)(Resistor, mcu));

[] [] true


[]


[ "the value is a 'cat.Chip', not an instance of the given schema" ]


Checked by its own schema, not the part's: a chip's `mount`, narrowed, is checked as a surface mount, and a
resistor's `ohms`, which a part has not, is checked too. A value of the wrong scalar is built, but reported:

In [7]:
const [soic, r1k] = store.transaction(() => [
  store.cat.Chip((c: any) => c.name("SOIC").mount((s: any) => s.pitch("1.27mm"))),
  store.cat.Resistor((r: any) => r.name("1k").ohms("1k").mount((m: any) => m.through((t: any) => t.pins(2n)))),
]);
console.log(Validators.Validate(store)(Part, soic));
console.log(Validators.Validate(store)(Part, r1k));

[ "cat.Chip#0.mount.pitch: expected float, got str" ]


[ "cat.Resistor#0.ohms: expected int, got str" ]


<a id="T11.4"></a>
## 11.4 · Values of any schema: a rating, whatever the datasheet gives

A rating is an int, a float or a text. A union would make each reader pick a branch name; a **value of any schema**,
`OfAny`, holds a value of one of the schemas it is restricted to, and is written bare where only one of them fits its
plain form. An unrestricted one, `S.OfAny((a: any) => a)`, takes a value of any schema, a scalar or a value of a
registered schema, and is written tagged by its schema's name. Another install gives every part both, and so every
resistor and chip too:

In [8]:
/** A part's rating, an int, a float or a text, and a note of any schema. */
function ratings(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("tutorial.catalog.ratings/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Rating: S.OfAny, Part: S.OfObject })
    .spec([
      (_: any) => _.Rating.name("cat.Rating").schemas([S.OfInt, S.OfFloat, S.OfStr]),
      (_: any) => _.Part.name("cat.Part").properties({ rating: _.Rating, note: S.OfAny((a: any) => a) }),
    ])]))]);
}

store.transaction(ratings);
store.transaction(() => {
  store.cat.Resistor(r10k, (r: any) => r.rating(0.25).note(store.cat.Through((t: any) => t.pins(2n))));
  store.cat.Chip(mcu, (c: any) => c.rating("3.3V").note("8-bit"));
});
console.log([...Chip.properties.keys()]);
console.log(Json.dumps(written(Resistor, r10k)));
console.log(Json.dumps(written(Chip, mcu)));

[ "name", "mount", "rating", "note" ]


{"name": "10k", "mount": {"through": {"pins": 2}}, "rating": 0.25, "note": {"cat.Through": {"pins": 2}}, "ohms": 10000}


{"name": "MCU", "mount": {"pitch": 0.5}, "rating": {"scalar": {"str": "3.3V"}}, "note": {"str": "8-bit"}}


The resistor's rating, a float, is written bare: of the three schemas, only a float's plain form is a number with a
fraction. The chip's, a text, is wrapped, `{"scalar": {"str": ...}}`, because a float's plain form may be text too
(`"NaN"`), so a bare text could be read two ways; the reader never guesses which. Each note is tagged: the standalone
value object by its schema's name, `cat.Through`, and the text by its scalar's, `str`.

A value of no schema the rating is restricted to is refused when written. A bool is no int:

In [9]:
store.transaction(() => {
  try {
    store.cat.Chip(mcu, (c: any) => c.rating(true));
  } catch (error) {
    if (!(error instanceof TypeError)) throw error;
    console.log(String(error));
  }
});

TypeError: property 'rating' does not hold a value of bool: it holds one of Schemas.OfInt, Schemas.OfFloat, Schemas.OfStr


<a id="T11.5"></a>
## 11.5 · Parameters: a bus as wide as each design needs

A bus's width is chosen where the bus is used: it is a **parameter** of the schema, as a type parameter is of a
generic class. Parameters belong to `Parametrics`, an extension installed before use. A schema of any kind declares a
set of them, each named, described, and for a value of a schema (`for_value`) or for an object of an object schema
(`for_object`). Binding them, and expressions over them, are mbse-expressions'
([CAP-112: parameters modeled as schema data](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-112.md)). `Parametrics.validate` checks a schema's parameters,
which its own `validate()` does not read:

In [10]:
store.transaction(Parametrics.install); // before any install that declares parameters

/** A bus, whose width each use of it binds. */
function buses(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("tutorial.catalog.buses/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Bus: S.OfObject })
    .spec([(_: any) => _.Bus.name("cat.Bus").ref(true).properties({ name: S.OfStr }).parameters([
      txn.Parametrics.OfParameters((ps: any) => ps.parameters({
        width: (d: any) => d.parameter((p: any) => p.description("How many bits").for_value([S.OfInt])),
      }))])])]))]);
}

store.transaction(buses);
const Bus = store.registered("cat.Bus");
const [declared] = [...new Builders.OfObject.Builder(Bus, false, store).parameters]; // its set, read through its builder
console.log([...declared.params.parameters].map((e: any) => [e.name.get(), e.parameter.description]), Parametrics.validate(store, Bus));
const Odd = store.transaction(() => store.Schemas.OfObject((o: any) => o.name("cat.Odd").parameters([store.Parametrics.OfParameters(
  (ps: any) => ps.parameters({ $w: (d: any) => d.parameter((p: any) => p) }))]))); // a reserved name, for nothing
console.log(Odd.validate());
for (const problem of Parametrics.validate(store, Odd)) console.log(problem);

[ [ "width", "How many bits" ] ] []


[]


name '$w' is reserved: names starting with '$' belong to the wire format


parameter '$w' is for neither a value nor an object: a parameter is for one


<a id="T11.6"></a>
## 11.6 · A store without the extension reads what it can

The catalog goes to a tool whose store never installed `Parametrics`. A store's snapshot (case study 12) writes its
schemas and its data; reading it, the tool's store makes every schema and object it can, and ignores the members that
only an extension it lacks would hold: the schemas' parameters. Give the reader an array, and it says what it ignored,
and why. Data is never ignored this way: a schema the snapshot defines, the reader makes.

In [11]:
const tool: any = new Builders.Store(); // no Parametrics
const ignored: Plain.Ignored[] = [];
Plain.FromPlain(tool).Snapshot(Plain.ToPlain(store).Snapshot(), ignored);
console.log(ignored.map((i) => i.reason));
console.log([...tool.registered("cat.Bus").properties.keys()], [...tool.singleton("cat.TheCatalog").parts].map((e: any) => `${e.part.name}`));

[
  "Schemas.OfParameter has no adjacency 'for_value'",
  "Schemas.OfObject has no adjacency 'parameters'",
  "Schemas.OfObject has no adjacency 'parameters'"
]


[ "name" ] [ "10k", "MCU" ]


<a id="T11.7"></a>
## 11.7 · Generated schemas: an install made from a table

A supplier sends each datasheet's fields as a table, a field's name and its scalar. An install is a function of the
transaction it is given, so it may build its spec from anything the program holds: here, a property for each row of
the table. Run again, it finds the schema by its name and writes nothing; given a longer table, it adds the new
properties. Schemas generated from a program's own classes work the same way (in Python, from its dataclasses).

In [12]:
const fields = new Map([["tolerance", "OfFloat"], ["package_code", "OfStr"], ["pins", "OfInt"]]); // the supplier's table

/** A datasheet's schema, generated from the supplier's table. */
function datasheet(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("tutorial.catalog.datasheet/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Datasheet: S.OfObject })
    .spec([(_: any) => _.Datasheet.name("cat.Datasheet").properties(new Map([...fields].map(([name, of]) => [name, S[of]])))])]))]);
}

store.transaction(datasheet);
store.transaction(datasheet); // run again: nothing written
const Datasheet = store.registered("cat.Datasheet");
console.log([...Datasheet.properties].map(([name, prop]: [string, any]) => [name, prop.schema.name]), Datasheet.validate());

[
  [ "tolerance", "Schemas.OfFloat" ],
  [ "package_code", "Schemas.OfStr" ],
  [ "pins", "Schemas.OfInt" ]
] []


## What you learned

- A refinement, `.refines([Base])`, has `Base`'s members and its own; its objects are `Base`'s too, filling its links
  and in its extent. A member added to `Base` later, the refinement has.
- An object of a refinement validates against `Base` too, by every validator, checked by its own schema: a member
  it narrows as narrowed, and its own members too.
- A refinement may redeclare a member, only narrower, before `.refines` or after it: to one of a union's branches'
  schemas, to a refinement, or to one of a value of any schema's schemas. `validate()` reports what it can't confirm.
- A value of any schema, `OfAny`, holds a value of one of its schemas, written bare where one alone fits its plain
  form and wrapped where it would be ambiguous; unrestricted, tagged by its schema. A value of another schema is
  refused when written.
- Parameters are `Parametrics`', installed before use: a set of them on a schema of any kind, each for a value or an
  object, checked by `Parametrics.validate`.
- A store lacking an extension reads a snapshot written with it, and lists what it ignored.
- An install is a function: it may generate its spec from a table, or anything else that describes the schemas.

**Next:** case study 12 keeps the catalog's history: who changed what, when and from where, and the journals and
snapshots that carry a store's data to another.

<!-- nav -->
[← 10 · Asking the store: mutations as queries](10_Asking_The_Store.ipynb) · [Home](../../README.md) · [12 · Keeping history: provenance, journals and snapshots →](12_Keeping_History.ipynb)